In [ ]:
import requests
api_key = "sk-ac80bb3b944f499db1229c51f5172e88"
# 1. 造一个天气工具的类
class WeatherTool:
    def __init__(self):
        # 给你的工具起个名字，这很重要，因为大模型要靠这个名字来调用它
        self.name = "get_weather"
        self.description = "查询南宁的真实天气情况"

    # 这就是工具的核心动作，大模型发指令后，代码就会执行这个函数
    def execute(self, args):
        # 请把你昨天写的那个请求 Open-Meteo 天气 API 的代码搬到这里来      
        # 提示：
        # 1. 定义 url
        # 2. 用 requests.get 发请求
        # 3. 提取温度
        # 4. return "南宁当前气温 wind = data["current_weather"]["windspeed"]XX 度"
        url = "https://api.open-meteo.com/v1/forecast?latitude=22.82&longitude=108.32&current_weather=true" 
        try: 
            res = requests.get(url, timeout=5)
            data = res.json()
            temp = data["current_weather"]["temperature"]
            wind = data["current_weather"]["windspeed"]
            return f"南宁当前气温 {temp}°C，风速 {wind} km/h。"
        except Exception as e:  
            return f"天气查询失败：{e}"

# ==========================================
# 测试你的黑盒
# ==========================================
weather = WeatherTool()
# 模拟大模型调用它，传个空字典作为参数
print(weather.execute({}))          

In [ ]:
import os
class KnowledgeTool:
    def __init__(self):
        self.name = "search_knowledge"
        self.decription = "查询本地知识库"
    def execute(self, args):
        query = args.get("query","")
        if not os.path.exists("knowledge.txt"):
            return "未找到知识库文件。"
        with open("knowledge.txt", "r", encoding="utf-8") as f:
            content = f.read()
        paragraphs = content.split("\n")   
        results = [p for p in paragraphs if any(query[i:i+2] in p for i in range(len(query) - 1))]
        if results:
            return "在知识库中找到了以下相关内容：\n" + "\n".join(results[:3])
        else:
            return "知识库中没有找到相关内容。"

                   

In [ ]:
# 实例化你的知识库工具
kb = KnowledgeTool()

# 模拟大模型发来指令，要查“迟到”
print(kb.execute({"query": "迟到"}))

In [3]:
import requests
import json
import os
import requests

class KnowledgeTool:
    def __init__(self):
        self.name = "search_knowledge"
        self.description = "查询本地知识库"
    def execute(self, args):
        query = args.get("query","")
        if not os.path.exists("knowledge.txt"):
            return "未找到知识库文件。"
        with open("knowledge.txt", "r", encoding="utf-8") as f:
            content = f.read()
        paragraphs = content.split("\n")   
        results = [p for p in paragraphs if any(query[i:i+2] in p for i in range(len(query) - 1))]
        if results:
            return "在知识库中找到了以下相关内容：\n" + "\n".join(results[:3])
        else:
            return "知识库中没有找到相关内容。"

class WeatherTool:
    def __init__(self):
        # 给你的工具起个名字，这很重要，因为大模型要靠这个名字来调用它
        self.name = "get_weather"
        self.description = "查询南宁的真实天气情况"

    # 这就是工具的核心动作，大模型发指令后，代码就会执行这个函数
    def execute(self, args):
        # 请把你昨天写的那个请求 Open-Meteo 天气 API 的代码搬到这里来      
        # 提示：
        # 1. 定义 url
        # 2. 用 requests.get 发请求
        # 3. 提取温度
        # 4. return "南宁当前气温 wind = data["current_weather"]["windspeed"]XX 度"
        url = "https://api.open-meteo.com/v1/forecast?latitude=22.82&longitude=108.32&current_weather=true" 
        try: 
            res = requests.get(url, timeout=5)
            data = res.json()
            temp = data["current_weather"]["temperature"]
            wind = data["current_weather"]["windspeed"]
            return f"南宁当前气温 {temp}°C，风速 {wind} km/h。"
        except Exception as e:  
            return f"天气查询失败：{e}"

class SuperAgent:
    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://api.deepseek.com/chat/completions"
        
        # 🌟 把咱们造好的两个黑盒装进“工具箱”里！
        self.tools = [
            WeatherTool(),
            KnowledgeTool()
        ]

    # 动态生成给大模型的“工具说明书”
    def get_tools_description(self):
        desc = "你可以使用以下工具。如果需要使用，请仅输出 JSON 格式：{\"tool\": \"工具名\", \"args\": {}}\n"
        for t in self.tools:
            desc += f"- {t.name}: {t.description}\n"
        return desc

    # 根据名字精准找到黑盒并执行
    def execute_tool(self, tool_name, args):
        for t in self.tools:
            if t.name == tool_name:
                return t.execute(args)
        return "未找到该工具"

    # 主脑逻辑：和昨天一模一样，但极度精简
    def chat(self, user_message):
        system_prompt = f"""你是一个智能助手。{self.get_tools_description()}
【铁律】：如果用户的问题涉及公司规定、扣钱、考勤、南宁美食等特定知识，你【必须】先调用 search_knowledge 工具去查本地知识库，绝对不允许直接凭自己的常识回答！
"""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json; charset=utf-8"
        }

        # 第一次请求
        data = {"model": "deepseek-chat", "messages": messages}
        response = requests.post(self.url, headers=headers, data=json.dumps(data, ensure_ascii=False).encode('utf-8'))
        ai_reply = response.json()["choices"][0]["message"]["content"]

        # 判断是否需要调用工具
        if "{" in ai_reply and "tool" in ai_reply:
            start = ai_reply.find("{")
            end = ai_reply.rfind("}") + 1
            try:
                tool_json = json.loads(ai_reply[start:end])
                tool_name = tool_json.get("tool")
                args = tool_json.get("args", {})
                
                # 执行黑盒
                tool_result = self.execute_tool(tool_name, args)
                print(f"🔧 [系统底层] 调用了工具：{tool_name}，结果：{tool_result}")
                
                # 第二次请求
                messages.append({"role": "assistant", "content": ai_reply})
                messages.append({"role": "user", "content": f"工具结果：{tool_result}。请根据此回答用户。"})
                
                data = {"model": "deepseek-chat", "messages": messages}
                final_response = requests.post(self.url, headers=headers, data=json.dumps(data, ensure_ascii=False).encode('utf-8'))
                return final_response.json()["choices"][0]["message"]["content"]
            except Exception as e:
                return f"解析工具失败：{e}"
        else:
            return ai_reply

# ==========================================
# 测试终极 Agent
# ==========================================
my_key = ""
agent = SuperAgent(my_key)

print("--- 测试问天气 ---")
print(agent.chat("南宁现在天气怎么样？"))

print("\n--- 测试问知识 ---")
print(agent.chat("迟到怎么扣钱？"))

--- 测试问天气 ---
🔧 [系统底层] 调用了工具：get_weather，结果：南宁当前气温 24.1°C，风速 10.7 km/h。
南宁当前气温 24.1°C，风速 10.7 km/h，天气比较舒适。

--- 测试问知识 ---
🔧 [系统底层] 调用了工具：search_knowledge，结果：未找到知识库文件。
抱歉，我暂时无法查询到公司关于迟到扣钱的具体规定（本地知识库中没有找到相关文件）。建议你直接咨询人事部门或查阅员工手册，以获取准确信息。
